# Word Embedding을 활용한 단어 찾기 게임

- Level 1. [Semantle](https://semantle.com/) 흉내내기
- Level 2. Semantle의 한국어 버전을 만들어보기
- Level 3. 단어 `A:B=C:?` 여기서 ? 맞추기 ([한국어 임베딩 확인](https://word2vec.kr/search/) + Semantle)

In [ ]:
#import gdown

#url = "https://drive.google.com/uc?id=1DCgLPJsfyLGZ99lB-aF8EvpKIWSZYgp4"
#output = 'data/ted_en.xml'

#gdown.download(url, output)

Downloading...
From (original): https://drive.google.com/uc?id=1DCgLPJsfyLGZ99lB-aF8EvpKIWSZYgp4
From (redirected): https://drive.google.com/uc?id=1DCgLPJsfyLGZ99lB-aF8EvpKIWSZYgp4&confirm=t&uuid=f33b2cc6-1588-4d8a-9139-b7cf7eb3a103
To: c:\SKN_24\nlp_practice\data\ted_en.xml
100%|██████████| 74.5M/74.5M [00:06<00:00, 10.9MB/s]


'data/ted_en.xml'

---

### naver 경제 기사 open api

#### 1. csv 전처리

In [1]:
# 경제 뉴스 파일 전철
import pandas as pd
import numpy as np

news_df=pd.read_csv('C:\\SKN_24\\nlp_practice\\data\\raw\\economy_news.csv')

# title+description=text
news_df['text']=news_df['title']+' '+news_df['description']

In [2]:
from konlpy.tag import Okt

okt = Okt()
ko_stopwords = ['은', '는', '이', '가', '을', '를', '와', '과', '들', '도',
                '부터', '까지', '에', '나', '너', '그', '것', '수', '등', '및']
preprocessed_data = []

In [3]:
# AssertionError 로 인해 추가
# NaN 제거 후 문자열 변환
news_df['text'] = news_df['text'].fillna('').astype(str)
news_df = news_df[news_df['text'].str.strip() != '']

print(f'유효 데이터: {len(news_df)}개')

유효 데이터: 6415개


#### level 1

In [39]:
from tqdm import tqdm
from konlpy.tag import Okt

for sentence in tqdm(news_df['text']):
    tokens=okt.morphs(sentence, stem=True)
    tokens = [t for t in tokens if t not in ko_stopwords and len(t) > 1]    
    preprocessed_data.append(tokens)

100%|██████████| 6415/6415 [00:18<00:00, 349.03it/s]


In [40]:
from gensim.models import Word2Vec
from gensim.models import KeyedVectors
model = Word2Vec(
    sentences=preprocessed_data,
    vector_size=100,
    window=5,
    min_count=2,   # 경제 뉴스는 데이터량이 적으니 2로 낮춤
    sg=1
)

In [6]:
# 학습된 모델로 게임 실행
wv = model.wv
semantle_game(wv)

NameError: name 'semantle_game' is not defined

In [41]:
import random
from datetime import date

def get_daily_word(wv):
    today_seed = int(date.today().strftime('%Y%m%d'))
    random.seed(today_seed)
    vocab = list(wv.index_to_key)
    return random.choice(vocab)

def semantle_game(wv):
    answer = get_daily_word(wv)
    history = []
    attempt = 0

    print('=' * 50)
    print('  🎮 한국어 Semantle 게임 시작!')
    print('  정답 단어를 맞춰보세요.')
    print('  유사도가 1.000에 가까울수록 정답에 가깝습니다.')
    print('  종료하려면 "quit" 입력')
    print('=' * 50)

    while True:
        guess = input(f'\n[{attempt+1}번째 시도] 단어 입력: ').strip()

        if guess == 'quit':
            print(f'\n게임 종료. 정답은 [{answer}] 이었습니다.')
            break

        if guess == answer:
            print(f'\n🎉 정답입니다! [{answer}] (시도 횟수: {attempt+1}번)')
            break

        if guess not in wv:
            print(f'  ⚠️  [{guess}]는 어휘에 없는 단어입니다.')
            continue

        score = float(wv.similarity(guess, answer))
        attempt += 1
        history.append((guess, score))

        if score >= 0.8:
            hint = '🔥 매우 가깝습니다!'
        elif score >= 0.6:
            hint = '♨️  꽤 가깝습니다!'
        elif score >= 0.4:
            hint = '🌡️  조금 가깝습니다.'
        else:
            hint = '❄️  멀습니다.'

        print(f'  유사도: {score:.4f}  {hint}')

        if history:
            print('\n  ⭐ 시도 기록 (유사도 순) ⭐')
            for word, sim in sorted(history, key=lambda x: x[1], reverse=True)[:5]:
                print(f'  {word:<10} {sim:.4f}')

In [42]:
# 학습된 모델로 게임 실행
wv = model.wv
semantle_game(wv)

  🎮 한국어 Semantle 게임 시작!
  정답 단어를 맞춰보세요.
  유사도가 1.000에 가까울수록 정답에 가깝습니다.
  종료하려면 "quit" 입력
  유사도: 0.2883  ❄️  멀습니다.

  ⭐ 시도 기록 (유사도 순) ⭐
  금리         0.2883
  유사도: 0.2404  ❄️  멀습니다.

  ⭐ 시도 기록 (유사도 순) ⭐
  금리         0.2883
  경제         0.2404
  유사도: 0.1839  ❄️  멀습니다.

  ⭐ 시도 기록 (유사도 순) ⭐
  금리         0.2883
  경제         0.2404
  시장         0.1839
  유사도: 0.1759  ❄️  멀습니다.

  ⭐ 시도 기록 (유사도 순) ⭐
  금리         0.2883
  경제         0.2404
  시장         0.1839
  주식         0.1759
  유사도: 0.2358  ❄️  멀습니다.

  ⭐ 시도 기록 (유사도 순) ⭐
  금리         0.2883
  경제         0.2404
  대통령        0.2358
  시장         0.1839
  주식         0.1759
  ⚠️  [ㅂㅂ]는 어휘에 없는 단어입니다.
  ⚠️  [ㅇㅇ]는 어휘에 없는 단어입니다.
  유사도: 0.1759  ❄️  멀습니다.

  ⭐ 시도 기록 (유사도 순) ⭐
  금리         0.2883
  경제         0.2404
  대통령        0.2358
  시장         0.1839
  주식         0.1759
  유사도: 0.1828  ❄️  멀습니다.

  ⭐ 시도 기록 (유사도 순) ⭐
  금리         0.2883
  경제         0.2404
  대통령        0.2358
  시장         0.1839
  수출         0.1828
  유사도: 0.5735  🌡️  조금 가깝습니다.

  ⭐ 시도 

---

### Level 2. 한국어 경제 Semantle

- Level 1에서 업그레이드된 버전
- **명사만** 정답 후보로 필터링 → 게임 품질 향상
- **순위 시스템** 추가 → 정답과 유사한 1000개 단어 내 순위 표시
- **hint** 입력 시 정답과 가장 유사한 단어 Top 5 공개

In [36]:
import random
from datetime import date

In [37]:
def get_daily_word_noun(wv, okt):
    """
    오늘 날짜를 시드로 사용해 명사 어휘에서 정답 단어 결정
    - 같은 날에는 항상 같은 단어가 정답
    - 빈도 상위 3000개 단어 중 명사만 후보로 사용
    """
    noun_vocab = []
    for word in wv.index_to_key[:3000]:
        pos = okt.pos(word)
        if pos and pos[0][1] == 'Noun' and len(word) >= 2:
            noun_vocab.append(word)

    print(f'명사 어휘 수: {len(noun_vocab)}개')

    today_seed = int(date.today().strftime('%Y%m%d'))
    random.seed(today_seed)
    return random.choice(noun_vocab)

In [ ]:
#Level 2 게임 함수 정의

def semantle_korean_game(wv, okt):
    """
    Level 2 Semantle 게임
    - 명사 필터링된 어휘에서 정답 선택하게 하기
    - 유사도 점수 + 순위 + 컬러바 표시가 뜸
    - hint 명령어로 Top 5 힌트 제공됨
    """
    print('명사 어휘 필터링')
    answer = get_daily_word_noun(wv, okt)

    # 정답 단어의 Top 1000 유사 단어 미리 계산 (순위 힌트용)
    top1000 = wv.most_similar(answer, topn=1000)
    rank_map = {word: rank+1 for rank, (word, _) in enumerate(top1000)}

    history = []
    attempt = 0

    print('=' * 55)
    print('  KR 한국어 경제 Semantle!')
    print('  오늘의 경제 단어를 맞추세요!!')
    print('  유사도: 0.000 ~ 1.000 (높을수록 정답에 가까움)')
    print('  순위: 정답과 유사한 1000개 단어 내 순위')
    print('  hint 입력 시 Top 5 힌트 공개')
    print('  quit 입력 시 종료')
    print('=' * 55)

    while True:
        guess = input(f'\n[{attempt+1}번째] 단어 입력: ').strip()

        if guess == 'quit':
            print(f'\n게임 종료. 정답은 [{answer}] 이었습니다.')
            break

        # 힌트 기능
        if guess == 'hint':
            print('  💡 [힌트] 정답과 유사한 단어 Top 5:')
            for i, (w, s) in enumerate(top1000[:5], 1):
                print(f'    {i}위: {w} (유사도: {s:.4f})')
            continue

        if guess == answer:
            print(f'\n🎉 정답! [{answer}] (시도: {attempt+1}번)')
            break

        if guess not in wv:
            print(f'  ⚠️  [{guess}]는 어휘에 없습니다.')
            continue

        score = float(wv.similarity(guess, answer))
        rank = rank_map.get(guess, '1000위 이하')
        attempt += 1
        history.append((guess, score, rank))

        # 유사도에 따른 컬러바
        if score >= 0.8:
            bar = '🟥🟥🟥🟥🟥'
        elif score >= 0.6:
            bar = '🟧🟧🟧🟧'
        elif score >= 0.4:
            bar = '🟨🟨🟨'
        elif score >= 0.2:
            bar = '🟩🟩'
        else:
            bar = '🟦'

        print(f'  유사도: {score:.4f}  순위: {rank}  {bar}')

        # 시도 기록 (유사도 높은 순 Top 5)
        print('\n  ⭐ 시도 기록 ⭐')
        for w, s, r in sorted(history, key=lambda x: x[1], reverse=True)[:5]:
            print(f'  {w:<10} 유사도: {s:.4f}  순위: {r}')

In [38]:
# Level 2 게임 실행
# 게임 종류 입력어: quit 
import random
from datetime import date

semantle_korean_game(wv, okt)

명사 어휘 필터링 중... (시간이 걸릴 수 있어요)
명사 어휘 수: 2242개
  🇰🇷 한국어 경제 Semantle!
  오늘의 경제 단어를 맞추세요.
  • 유사도: 0.000 ~ 1.000 (높을수록 정답에 가까움)
  • 순위: 정답과 유사한 1000개 단어 내 순위
  • hint 입력 시 Top 5 힌트 공개
  • quit 입력 시 종료
  유사도: 0.2776  순위: 1000위 이하  🟩🟩

  --- 시도 기록 ---
  금리         유사도: 0.2776  순위: 1000위 이하
  유사도: 0.4701  순위: 1000위 이하  🟨🟨🟨

  --- 시도 기록 ---
  금융         유사도: 0.4701  순위: 1000위 이하
  금리         유사도: 0.2776  순위: 1000위 이하
  유사도: 0.4280  순위: 1000위 이하  🟨🟨🟨

  --- 시도 기록 ---
  금융         유사도: 0.4701  순위: 1000위 이하
  사전         유사도: 0.4280  순위: 1000위 이하
  금리         유사도: 0.2776  순위: 1000위 이하
  유사도: 0.4323  순위: 1000위 이하  🟨🟨🟨

  --- 시도 기록 ---
  금융         유사도: 0.4701  순위: 1000위 이하
  대화         유사도: 0.4323  순위: 1000위 이하
  사전         유사도: 0.4280  순위: 1000위 이하
  금리         유사도: 0.2776  순위: 1000위 이하
  유사도: 0.2517  순위: 1000위 이하  🟩🟩

  --- 시도 기록 ---
  금융         유사도: 0.4701  순위: 1000위 이하
  대화         유사도: 0.4323  순위: 1000위 이하
  사전         유사도: 0.4280  순위: 1000위 이하
  금리         유사도: 0.2776  순위: 1000위 이하
  경제     

---

In [ ]:
# 단어 유추 함수 정의

def word_analogy(wv, word_a, word_b, word_c, topn=5):
    """
    단어 유추: A:B = C:?
    - 벡터 연산: vec(B) - vec(A) + vec(C)
    - positive: B, C 방향으로 이동됨
    - negative: A 방향에서 멀어짐

    예시:word_analogy(wv, '미국', '달러', '일본') → '엔' 예측
    """
    for word in [word_a, word_b, word_c]:
        if word not in wv:
            print(f'⚠️  [{word}]는 어휘에 없습니다.')
            return None

    results = wv.most_similar(
        positive=[word_b, word_c],   # + vec(B) + vec(C)
        negative=[word_a],           # - vec(A)
        topn=topn
    )

    print(f'\n  {word_a} : {word_b} = {word_c} : ?')
    print('  ' + '-' * 30)
    for rank, (word, score) in enumerate(results, 1):
        print(f'  {rank}위: {word:<12} (유사도: {score:.4f})')

    return results

In [35]:
# 단어 유추 예시 확인
# 경제 도메인 단어로 벡터 연산 결과 확인

word_analogy(wv, '미국', '달러', '일본')   # 예상: 엔


  미국 : 달러 = 일본 : ?
  ------------------------------
  1위: 돌다           (유사도: 0.6710)
  2위: 11.7%        (유사도: 0.6595)
  3위: 49억          (유사도: 0.6546)
  4위: 9억           (유사도: 0.6532)
  5위: 386억         (유사도: 0.6515)


[('돌다', 0.670981764793396),
 ('11.7%', 0.659505307674408),
 ('49억', 0.6546359658241272),
 ('9억', 0.6532440185546875),
 ('386억', 0.6514851450920105)]

In [43]:
# Level 3 게임 함수 정의
def word_analogy_game(wv):
    """
    A:B = C:? 게임 루프
    - 경제 도메인 퀴즈 5문제 나옴
    - 직접 정답 or 모델 예측하고 Top 3 포함하면 정답으로 인정
    - hint 입력하는 경우 모델 예측 Top 3 공개해줌
    """

    quiz_game=[
        ('미국','달러','일본','엔'),
        ('금리','인상','물가','상승'),
        ('수출','증가','확대','수입'),
        ('미국주식','배당','임대','부동산'),
        ('코스피','상승','하락','환율')
    ]

In [44]:
print('='*55)
print('단어 유추 게임: A : B = C : ?')
print('빈칸 ?에 들어갈 단어를 맞추세요!')
print('hint 입력 시 모델 예측 Top 3 공개')
print('quit 입력 시 종료')
print('='*55)
score = 0

단어 유추 게임: A : B = C : ?
빈칸 ?에 들어갈 단어를 맞추세요!
hint 입력 시 모델 예측 Top 3 공개
quit 입력 시 종료


In [51]:
for i, (word_a, word_b, word_c, answer) in enumerate(quiz_game):
    print(f'\n[문제 {i+1}]  {word_a} : {word_b} = {word_c} : ?')

NameError: name 'quiz_game' is not defined

In [54]:
        try:
            predictions = wv.most_similar(
                positive=[word_b, word_c],
                negative=[word_a],
                topn=10
            )
            pred_words = [w for w, _ in predictions]
        except:
            pred_words = []

        while True:
            guess = input('  정답 입력: ').strip()

            if guess == 'quit':
                print(f'\n게임 종료. 최종 점수: {score}/{len(quiz_bank)}')
                return

            if guess == 'hint':
                print(f'  💡 모델 예측 Top 3: {pred_words[:3]}')
                continue


SyntaxError: 'return' outside function (3007149620.py, line 16)

In [57]:
# Level 3 게임 함수 정의
def word_analogy_game(wv):
    """
    A:B = C:? 게임 루프
    - 경제 도메인 퀴즈 5문제 나옴
    - 직접 정답 or 모델 예측하고 Top 3 포함하면 정답으로 인정
    - hint 입력하는 경우 모델 예측 Top 3 공개해줌
    """

    quiz_game=[
        ('미국','달러','일본','엔'),
        ('금리','인상','물가','상승'),
        ('수출','증가','확대','수입'),
        ('미국주식','배당','임대','부동산'),
        ('코스피','상승','하락','환율')
    ]
    print('='*55)
    print('단어 유추 게임: A : B = C : ?')
    print('빈칸 ?에 들어갈 단어를 맞추세요!')
    print('hint 입력 시 모델 예측 Top 3 공개')
    print('quit 입력 시 종료')
    print('='*55)
    score = 0

    for i, (word_a, word_b, word_c, answer) in enumerate(quiz_game):
        print(f'\n[문제 {i+1}]  {word_a} : {word_b} = {word_c} : ?')
        # 모델 예측 결과 미리 계산
        try:
            predictions = wv.most_similar(
                positive=[word_b, word_c],
                negative=[word_a],
                topn=10
            )
            pred_words = [w for w, _ in predictions]
        except:
            pred_words = []

        while True:
            guess = input('  정답 입력: ').strip()

            if guess == 'quit':
                print(f'\n게임 종료. 최종 점수: {score}/{len(quiz_bank)}')
                return

            if guess == 'hint':
                print(f'  💡 모델 예측 Top 3: {pred_words[:3]}')
                continue

            # 정답 판정: 직접 정답 OR 모델 Top 3 내 포함
            if guess == answer or guess in pred_words[:3]:
                print(f'  ✅ 정답! (모범 답안: {answer})')
                score += 1
            else:
                print(f'  ❌ 틀렸습니다. (정답: {answer})')
                print(f'     모델 예측 Top 5: {pred_words[:5]}')
            break

    print(f'\n🏆 최종 점수: {score}/{len(quiz_game)}')

In [58]:
# Level 3 게임 실행

word_analogy_game(wv)

단어 유추 게임: A : B = C : ?
빈칸 ?에 들어갈 단어를 맞추세요!
hint 입력 시 모델 예측 Top 3 공개
quit 입력 시 종료

[문제 1]  미국 : 달러 = 일본 : ?
  ✅ 정답! (모범 답안: 엔)

[문제 2]  금리 : 인상 = 물가 : ?
  ❌ 틀렸습니다. (정답: 상승)
     모델 예측 Top 5: ['기름값', '압박', '식료품', '소고기', '엄단']

[문제 3]  수출 : 증가 = 확대 : ?
  ✅ 정답! (모범 답안: 수입)

[문제 4]  미국주식 : 배당 = 임대 : ?
  ✅ 정답! (모범 답안: 부동산)

[문제 5]  코스피 : 상승 = 하락 : ?
  ✅ 정답! (모범 답안: 환율)

🏆 최종 점수: 4/5
